# Chapter 2 · Foundations in Rust

This original companion uses **nevo** to make the evolutionary and neural building blocks executable. It follows the [official Chapter 2 outline](https://neuroevolutionbook.com/) and cites the methods in [the written chapter](../book/ch02_foundations.md). It is an independently specified implementation, not an official exercise solution or a reproduction of published benchmark results.

Run all cells in order using the Rust/Evcxr kernel, with the working directory set to `notebooks/`. The library uses scalar `f64` Rust code; the only plotting dependency here is Plotters. First-run dependency compilation may take a few minutes.

In [1]:
:dep nevo = { path = "../crates/nevo" }
:dep rand_chacha = "=0.9.0"
:dep plotters = { version = "=0.3.7", default-features = false, features = ["evcxr", "line_series"] }
use nevo::{Algorithm, Evaluated, Evaluator, step};
use nevo::algorithms::*;
use nevo::benchmarks::{Sphere, Rastrigin, Zdt1};
use nevo::nn::*;
use nevo::tasks::Xor;
use plotters::prelude::*;

## 1. Generate → Evaluate → Select

`generate` issues candidate jobs; the evaluator produces typed results; `select` incorporates those results into the next state. Selection can mean retaining parents, adapting a distribution, or preserving a Pareto front. There are still only two core traits.

Here is a deliberately explicit runner for scalar minimization. It counts every objective call and keeps the best **observed** score; it never evaluates a distribution mean secretly. Production examples can use `nevo::step` instead. This teaching helper assumes the supplied algorithms honor the batch contract; `step` additionally checks empty batches and duplicate job IDs.

In [2]:
fn initial_mean(dimensions: usize, seed: u64, bound: f64) -> Vec<f64> {
    use rand_chacha::rand_core::{RngCore, SeedableRng};
    let mut rng = rand_chacha::ChaCha8Rng::seed_from_u64(seed);
    rng.set_stream(2); // Independent from search stream 0 and evaluation stream 1.
    (0..dimensions).map(|_| {
        let u = (rng.next_u64() >> 11) as f64 / 9_007_199_254_740_992.0;
        bound * (2.0 * u - 1.0)
    }).collect()
}

fn trace<A, E>(mut algorithm: A, mut evaluator: E, batches: usize)
    -> Result<Vec<(usize, f64)>, Box<dyn std::error::Error>>
where A: Algorithm<Input=Vec<f64>, Output=f64>,
      E: Evaluator<Vec<f64>, Output=f64>
{
    let mut evaluations = 0;
    let mut best = f64::INFINITY;
    let mut history = Vec::new();
    for _ in 0..batches {
        let jobs = algorithm.generate()?;
        let mut results = Vec::with_capacity(jobs.len());
        for job in jobs {
            let output = evaluator.evaluate(&job.input, job.seed)?;
            evaluations += 1;
            best = best.min(output);
            results.push(Evaluated { job, output });
        }
        algorithm.select(results)?;
        history.push((evaluations, best));
    }
    Ok(history)
}

## 2. Four scalar searches on Sphere

The GA uses tournament selection, uniform crossover, Gaussian mutation and one reevaluated elite. The `(μ,λ)` ES carries one self-adaptive mutation scale per individual and retains only offspring as the next parents. CMA-ES adapts a full covariance matrix and global scale. OpenAI-style ES uses mirrored perturbations, centered average-tie ranks, fixed noise and an explicitly scaled SGD update.

This live demonstration uses 8 dimensions, seed 42, 64 evaluations per batch and 100 batches including initialization. GA and ES have exactly the same initial uniform population. CMA-ES and OpenAI-style ES share a separately sampled uniform center, then sample Gaussian perturbations around it; their initial distributions differ from the GA/ES populations. The center uses ChaCha8 stream 2, independent of search stream 0 and evaluation-seed stream 1. All four receive 6,400 objective calls. The 20-seed experiment below is more informative than this single run.

In [3]:
let dimensions = 8;
let starting_mean = initial_mean(dimensions, 42, 5.0);
let mut cma_config = CmaEsConfig::new(starting_mean.clone(), 42);
cma_config.population = 64;
let histories = vec![
    ("GA", trace(Ga::new(GaConfig::new(dimensions, 42))?, Sphere::new(dimensions)?, 100)?),
    ("Self-adaptive ES", trace(Es::new(EsConfig::new(dimensions, 42))?, Sphere::new(dimensions)?, 100)?),
    ("CMA-ES", trace(CmaEs::new(cma_config)?, Sphere::new(dimensions)?, 100)?),
    ("OpenAI-style ES", trace(OpenAiEs::new(OpenAiEsConfig::new(starting_mean, 42))?, Sphere::new(dimensions)?, 100)?),
];
for (name, history) in &histories {
    let (count, best) = history.last().unwrap();
    assert_eq!(*count, 6400);
    println!("{name:18} evaluations={count}, best_observed={best:.9e}");
}

GA                 evaluations=6400, best_observed=6.289540766e-5


Self-adaptive ES   evaluations=6400, best_observed=1.105372525e-9


CMA-ES             evaluations=6400, best_observed=8.229501164e-13


OpenAI-style ES    evaluations=6400, best_observed=4.612102170e-1


()

In [4]:
let sphere_figure = evcxr_figure((900, 460), |root| {
    root.fill(&WHITE)?;
    let low = histories.iter().flat_map(|(_, h)| h.iter().map(|(_, y)| *y))
        .filter(|y| *y > 0.0).fold(1.0, f64::min) * 0.3;
    let mut chart = ChartBuilder::on(&root)
        .caption("Sphere · seed 42 · equal objective-call budgets", ("sans-serif", 22))
        .margin(18).x_label_area_size(44).y_label_area_size(70)
        .build_cartesian_2d(0u32..6400u32, (low..200.0f64).log_scale())?;
    chart.configure_mesh().x_desc("Objective evaluations").y_desc("Best observed objective (log)")
        .y_label_formatter(&|value| format!("{value:.0e}")).draw()?;
    for ((name, history), color) in histories.iter().zip([BLUE, RGBColor(0,140,120), RED, RGBColor(140,70,180)]) {
        chart.draw_series(LineSeries::new(history.iter().map(|(x,y)| (*x as u32, *y)), color.stroke_width(2)))?
            .label(*name).legend(move |(x,y)| PathElement::new(vec![(x,y),(x+20,y)], color.stroke_width(2)));
    }
    chart.configure_series_labels().background_style(WHITE.mix(0.95)).border_style(BLACK).draw()?;
    root.present()?;
    Ok(())
});
sphere_figure

Sphere · seed 42 · equal objective-call budgets
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Best observed objective (log)
 
 
Objective evaluations
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
1e-12
 
 
 
1e-10
 
 
 
1e-8
 
 
 
1e-6
 
 
 
1e-4
 
 
 
1e-2
 
 
 
1e0
 
 
 
1e2
 
 
 
 
0
 
 
 
1000
 
 
 
2000
 
 
 
3000
 
 
 
4000
 
 
 
5000
 
 
 
6000
 
 
<polyline fill="none" opacity="1" stroke="#0000FF" stroke-width="2" points="95,68 103,70 111,71 119,78 127,81 135,84 143,85 151,85 159,86 167,90 175,90 183,91 191,91 199,92 206,93 214,94 222,98 230,98 238,100 246,100 254,101 262,103 270,104 278,105 286,107 294,107 302,113 310,114 317,115 325,115 333,120 341,121 349,121 357,128 365,128 373,133 381,138 389,140 397,142 405,145 413,145 421,148 428,148 436,153 444,153 452,160 460,160 468,160 476,162 484,169 492,169 500,172 508,172 516,173 524,177 532,179 540,179 547,179 555,179 563,179 571,179 579,179 587,180 595,180 603,180 611,181 619,181 627,181 635,181 643,181 651,181 658,181 666,181 674,181 682,182 690,184 698,184 706,187 714,187 722,187 730,187 738,187 746,187 754,187 762,187 769,187 777,187 785,192 793,192 801,192 809,192 817,192 825,192 833,195 841,195 849,199 857,199 865,199 873,199 881,199 "/>
<polyline fill="none" opacity="1" stroke="#008C78" stroke-width="2" points="95,68 103,68 111,71 119,75 127,76 135,77 143,83 151,83 159,83 167,87 175,87 183,89 191,92 199,96 206,99 214,101 222,105 230,106 238,115 246,115 254,127 262,127 270,127 278,127 286,127 294,127 302,127 310,127 317,127 325,130 333,133 341,138 349,138 357,140 365,144 373,145 381,147 389,150 397,152 405,157 413,160 421,163 428,163 436,168 444,168 452,168 460,173 468,173 476,173 484,173 492,174 500,176 508,187 516,187 524,187 532,189 540,190 547,191 555,194 563,197 571,199 579,203 587,206 595,207 603,210 611,214 619,217 627,217 635,226 643,230 651,232 658,236 666,240 674,243 682,243 690,246 698,247 706,252 714,252 722,252 730,256 738,260 746,261 754,261 762,266 769,267 777,269 785,275 793,276 801,281 809,281 817,285 825,292 833,292 841,292 849,300 857,303 865,304 873,311 881,311 "/>
<polyline fill="none" opacity="1" stroke="#FF0000" stroke-width="2" points="95,65 103,70 111,78 119,85 127,85 135,85 143,85 151,85 159,94 167,94 175,98 183,98 191,98 199,108 206,119 214,119 222,119 230,119 238,128 246,128 254,128 262,128 270,133 278,141 286,141 294,147 302,147 310,147 317,147 325,147 333,154 341,157 349,160 357,160 365,161 373,168 381,168 389,174 397,178 405,185 413,188 421,189 428,192 436,198 444,202 452,206 460,210 468,216 476,217 484,220 492,221 500,228 508,228 516,235 524,244 532,244 540,245 547,245 555,255 563,263 571,263 579,263 587,273 595,273 603,273 611,274 619,277 627,279 635,287 643,292 651,292 658,306 666,306 674,306 682,306 690,311 698,311 706,311 714,311 722,315 730,315 738,320 746,329 754,330 762,335 769,338 777,342 785,348 793,348 801,354 809,357 817,359 825,370 833,382 841,383 849,383 857,383 865,385 873,385 881,385 "/>
<polyline fill="none" opacity="1" stroke="#8C46B4" stroke-width="2" points="95,66 103,66 111,66 119,66 127,67 135,67 143,67 151,67 159,67 167,69 175,69 183,69 191,69 199,70 206,70 214,70 222,70 230,71 238,71 246,71 254,73 262,73 270,73 278,73 286,73 294,73 302,73 310,73 317,73 325,73 333,74 341,75 349,75 357,75 365,75 373,75 381,75 389,76 397,76 405,76 413,78 421,78 428,78 436,79 444,79 452,79 460,80 468,80 476,83 484,83 492,83 500,83 508,83 516,83 524,83 532,83 540,83 547,86 555,86 563,86 571,86 579,86 587,87 595,87 603,87 611,89 619,89 627,89 635,96 643,96 651,96 658,96 666,96 674,96 682,96 690,96 698,96 706,96 714,100 722,100 730,100 738,104 746,104 754,104 762,104 769,104 777,104 785,104 793,104 801,104 809,104 817,104 825,104 833,104 841,104 849,104 857,106 865,106 873,106 881,108 "/>
 
 
 
GA
 
 
Self-adaptive ES
 
 
CMA-ES
 
 
OpenAI-style ES

### Inspect the recorded 20-seed experiment

The repository includes all runs, not only a selected successful seed. Recompute them with:

```sh
cargo run -p nevo --example ch02_search --release --locked -- book/results/ch02
```

The next cell reads the resulting CSV and reports median and interquartile range using linearly interpolated sample quantiles. The settings were held fixed across seeds 0–19. Rastrigin adds many local optima. A small objective value on Sphere does not establish broad optimizer superiority. OpenAI-style ES's fixed perturbation scale also separates its best sampled candidate from its generally unevaluated mean.

In [5]:
fn quantile(values: &[f64], p: f64) -> f64 {
    let mut sorted = values.to_vec();
    sorted.sort_by(f64::total_cmp);
    let position = p * (sorted.len() - 1) as f64;
    let lower = position.floor() as usize;
    let upper = position.ceil() as usize;
    sorted[lower] + (sorted[upper] - sorted[lower]) * (position - lower as f64)
}
let csv = std::fs::read_to_string("../book/results/ch02/search_runs.csv")?;
let mut runs = std::collections::BTreeMap::<(String, String), Vec<f64>>::new();
for line in csv.lines().skip(1) {
    let columns: Vec<_> = line.split(',').collect();
    assert_eq!(columns.len(), 7);
    assert_eq!(columns[3].parse::<usize>()?, 6400);
    runs.entry((columns[0].to_owned(), columns[1].to_owned())).or_default()
        .push(columns[5].parse::<f64>()?);
}
for ((problem, algorithm), values) in &runs {
    assert_eq!(values.len(), 20);
    println!("{problem:10} {algorithm:10}: median={:.5e}, IQR=[{:.5e}, {:.5e}], target hits={}/20",
        quantile(values,0.5),quantile(values,0.25),quantile(values,0.75),
        values.iter().filter(|&&v| v <= 1e-8).count());
}

rastrigin  cma_es    : median=2.98496e0, IQR=[1.98992e0, 3.97994e0], target hits=0/20


rastrigin  es        : median=1.98991e1, IQR=[1.49244e1, 2.21378e1], target hits=0/20


rastrigin  ga        : median=5.47814e0, IQR=[2.99673e0, 7.22265e0], target hits=0/20


rastrigin  openai_es : median=5.37675e1, IQR=[4.70431e1, 6.13675e1], target hits=0/20


sphere     cma_es    : median=5.04538e-13, IQR=[1.62698e-13, 1.21168e-12], target hits=20/20


sphere     es        : median=8.69707e-10, IQR=[4.45031e-10, 1.78217e-9], target hits=19/20


sphere     ga        : median=8.36575e-5, IQR=[4.94485e-5, 1.16055e-4], target hits=0/20


sphere     openai_es : median=1.69518e0, IQR=[1.31218e0, 4.80250e0], target hits=0/20


()

## 3. Multiobjective selection without changing the core traits

ZDT1 returns two minimization objectives. NSGA-II combines parents with offspring and ranks them by Pareto dominance, using crowding distance to preserve spread. This version uses bounded uniform crossover and Gaussian mutation; it does not implement the paper's SBX/polynomial variation or constraint-domination extension. Parent scores are cached, so this implementation assumes a stationary deterministic task.

We use the original 30-dimensional ZDT1 form, population 64 and 200 batches. The analytic front is `f₂ = 1 − √f₁` when genes 2–30 are all zero. The curve below is a mathematical reference, not injected into the population.

In [6]:
let mut pareto = Nsga2::new(Nsga2Config::new(30, vec![Objective::Minimize; 2], 42))?;
let mut zdt = Zdt1::new(30)?;
let mut initial_points = Vec::new();
for batch in 1..=200 {
    step(&mut pareto, &mut zdt)?;
    if batch == 1 { initial_points = pareto.parents().iter().map(|r| (r.output[0],r.output[1])).collect::<Vec<_>>(); }
}
let front_points: Vec<_> = pareto.pareto_front().iter().map(|r| (r.output[0],r.output[1])).collect();
println!("evaluations={}, retained={}, nondominated={}", pareto.evaluations(), pareto.parents().len(), front_points.len());
let vertical_gap = front_points.iter().map(|(x,y)| y - (1.0-x.sqrt())).sum::<f64>() / front_points.len() as f64;
println!("mean vertical distance above the reference curve: {vertical_gap:.9e}");

evaluations=12800, retained=64, nondominated=64


mean vertical distance above the reference curve: 9.464360436e-5


In [7]:
let pareto_figure = evcxr_figure((900, 440), |root| {
    root.fill(&WHITE)?;
    let y_max = initial_points.iter().map(|(_,y)| *y).fold(1.0,f64::max) * 1.1;
    let mut chart = ChartBuilder::on(&root).caption("ZDT1 · 30 genes · seed 42", ("sans-serif",22))
        .margin(18).x_label_area_size(42).y_label_area_size(50)
        .build_cartesian_2d(-0.03f64..1.03f64, -0.1f64..y_max)?;
    chart.configure_mesh().x_desc("f1 (minimize)").y_desc("f2 (minimize)").draw()?;
    chart.draw_series(initial_points.iter().map(|&(x,y)| Circle::new((x,y),3,BLACK.mix(0.25).filled())))?.label("Initial population")
        .legend(|(x,y)| Circle::new((x,y),3,BLACK.mix(0.25).filled()));
    chart.draw_series(front_points.iter().map(|&(x,y)| Circle::new((x,y),3,BLUE.filled())))?.label("Final nondominated parents")
        .legend(|(x,y)| Circle::new((x,y),3,BLUE.filled()));
    chart.draw_series(LineSeries::new((0..=200).map(|i| { let x=i as f64/200.0; (x,1.0-x.sqrt()) }),RED.stroke_width(2)))?.label("Analytic Pareto front")
        .legend(|(x,y)| PathElement::new(vec![(x,y),(x+20,y)],RED.stroke_width(2)));
    chart.configure_series_labels().background_style(WHITE.mix(0.95)).border_style(BLACK).draw()?;
    root.present()?;
    Ok(())
});
pareto_figure

ZDT1 · 30 genes · seed 42
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
f2 (minimize)
 
 
f1 (minimize)
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
0.0
 
 
 
1.0
 
 
 
2.0
 
 
 
3.0
 
 
 
4.0
 
 
 
5.0
 
 
 
6.0
 
 
 
 
0.0
 
 
 
0.1
 
 
 
0.2
 
 
 
0.3
 
 
 
0.4
 
 
 
0.5
 
 
 
0.6
 
 
 
0.7
 
 
 
0.8
 
 
 
0.9
 
 
 
1.0
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
<polyline fill="none" opacity="1" stroke="#FF0000" stroke-width="2" points="91,320 94,324 98,325 102,326 106,328 110,328 114,329 117,330 121,331 125,331 129,332 133,333 137,333 140,334 144,334 148,335 152,335 156,336 160,336 163,337 167,337 171,337 175,338 179,338 183,339 186,339 190,339 194,340 198,340 202,341 206,341 209,341 213,342 217,342 221,342 225,343 229,343 232,343 236,343 240,344 244,344 248,344 252,345 255,345 259,345 263,346 267,346 271,346 275,346 278,347 282,347 286,347 290,348 294,348 298,348 301,348 305,349 309,349 313,349 317,349 321,350 324,350 328,350 332,350 336,351 340,351 344,351 347,351 351,351 355,352 359,352 363,352 367,352 370,353 374,353 378,353 382,353 386,353 390,354 393,354 397,354 401,354 405,355 409,355 413,355 416,355 420,355 424,356 428,356 432,356 436,356 439,356 443,357 447,357 451,357 455,357 459,357 462,358 466,358 470,358 474,358 478,358 482,359 486,359 489,359 493,359 497,359 501,360 505,360 509,360 512,360 516,360 520,360 524,361 528,361 532,361 535,361 539,361 543,362 547,362 551,362 555,362 558,362 562,362 566,363 570,363 574,363 578,363 581,363 585,363 589,364 593,364 597,364 601,364 604,364 608,364 612,365 616,365 620,365 624,365 627,365 631,365 635,366 639,366 643,366 647,366 650,366 654,366 658,366 662,367 666,367 670,367 673,367 677,367 681,367 685,368 689,368 693,368 696,368 700,368 704,368 708,369 712,369 716,369 719,369 723,369 727,369 731,369 735,370 739,370 742,370 746,370 750,370 754,370 758,370 762,371 765,371 769,371 773,371 777,371 781,371 785,371 788,372 792,372 796,372 800,372 804,372 808,372 811,372 815,373 819,373 823,373 827,373 831,373 834,373 838,373 842,374 846,374 850,374 854,374 857,374 "/>
 
 
 
Initial population
 
 
Final nondominated parents
 
 
Analytic Pareto front

## 4. Neural parameters are ordinary genomes

An MLP stores a flat parameter vector. In each layer, each output neuron stores its input weights followed by its bias. A `2 → 4 → 1` network therefore has `(2+1)×4 + (4+1)×1 = 17` parameters. The same network can be evaluated by an evolutionary algorithm or trained by its analytic MSE gradient.

First check a one-neuron linear example by hand: with weights `[2, −1]`, bias `0.5`, input `[3,4]`, the output is `2.5`. Gradients are tested against central finite differences in `tests/neural.rs`.

In [8]:
let affine = Mlp::new(&[2,1], &[Activation::Identity], vec![2.0,-1.0,0.5])?;
assert_eq!(affine.forward(&[3.0,4.0])?, vec![2.5]);
let data = vec![TrainingExample { input:vec![2.0], target:vec![3.0] }, TrainingExample { input:vec![1.0], target:vec![1.0] }];
let mut linear = Mlp::new(&[1,1], &[Activation::Identity], vec![0.0,0.0])?;
let (loss, gradient) = linear.mse_gradient(&data)?;
println!("MSE={loss}, gradient={gradient:?}");
linear.train_step(&data,0.1)?;
println!("After one gradient step: parameters={:?}, MSE={}", linear.parameters(),linear.mse(&data)?);

MSE=5, gradient=[-7.0, -4.0]


After one gradient step: parameters=[0.7000000000000001, 0.4], MSE=0.7249999999999996


### Evolve XOR weights

This uses every row of the XOR truth table. It is a representation and optimization exercise, not a held-out generalization test. A fixed random initial mean uses ChaCha8 stream 2 and uniform coordinates in `[-0.5,0.5)`, exactly as in the executable example. This initialization stream is distinct from the search perturbation stream. Both runs stop once MSE is at most `0.001`. CMA-ES is capped at 400 batches of 32 candidates; backpropagation uses the same starting weights and a cap of 5,000 full-batch gradient steps. This explicit stopping rule avoids pursuing sigmoid saturation just to print smaller losses. Those work units differ; the panels below are not an equal-cost speed comparison.

In [9]:
let mut xor = Xor::new(4)?;
let initial_weights = initial_mean(xor.dimensions(),42,0.5);
let initial_xor_loss = xor.value(&initial_weights)?;
let mut learned = xor.network(initial_weights.clone())?;
let mut xor_config = CmaEsConfig::new(initial_weights,42);
xor_config.population = 32;
let mut xor_search = CmaEs::new(xor_config)?;
let mut evolution_curve = vec![(0usize,initial_xor_loss)];
for _ in 0..400 {
    step(&mut xor_search,&mut xor)?;
    evolution_curve.push((xor_search.evaluations(),xor_search.best().unwrap().output));
    if xor_search.best().unwrap().output <= 0.001 { break; }
}
let mut gradient_curve = vec![(0usize,initial_xor_loss)];
let mut gradient_steps = 0;
for update in 1..=5000 {
    learned.train_step(xor.data(),1.0)?;
    gradient_steps = update;
    let loss = learned.mse(xor.data())?;
    if update % 10 == 0 || loss <= 0.001 { gradient_curve.push((update,loss)); }
    if loss <= 0.001 { break; }
}
let evolved = xor.network(xor_search.best().unwrap().job.input.clone())?;
println!("CMA-ES: {} search evaluations, MSE={:.9e}",xor_search.evaluations(),xor_search.best().unwrap().output);
println!("Backprop: {gradient_steps} full-batch steps, MSE={:.9e}",learned.mse(xor.data())?);
for row in xor.data() {
    println!("{:?} -> target {}, evolved {:.6}, backprop {:.6}",row.input,row.target[0],evolved.forward(&row.input)?[0],learned.forward(&row.input)?[0]);
}

CMA-ES: 1056 search evaluations, MSE=5.741047324e-4


Backprop: 1100 full-batch steps, MSE=9.990828383e-4


[0.0, 0.0] -> target 0, evolved 0.000024, backprop 0.031454


[0.0, 1.0] -> target 1, evolved 0.965301, backprop 0.971286


[1.0, 0.0] -> target 1, evolved 0.966961, backprop 0.970058


[1.0, 1.0] -> target 0, evolved 0.000908, backprop 0.035861


()

In [10]:
let xor_figure = evcxr_figure((1000,420),|root| {
    root.fill(&WHITE)?;
    let areas = root.split_evenly((1,2));
    for (area,(title,label,history,color)) in areas.iter().zip([
        ("Evolved XOR network","Objective evaluations",&evolution_curve,BLUE),
        ("Differentiated XOR network","Full-batch gradient steps",&gradient_curve,RGBColor(0,140,120)),
    ]) {
        let low = history.iter().map(|(_,v)| *v).filter(|v| *v > 0.0).fold(0.1,f64::min) * 0.3;
        let mut chart=ChartBuilder::on(area).caption(title,("sans-serif",20)).margin(14)
            .x_label_area_size(42).y_label_area_size(66)
            .build_cartesian_2d(0u32..history.last().unwrap().0 as u32,(low..1.0f64).log_scale())?;
        chart.configure_mesh().x_desc(label).y_desc("Truth-table MSE (log)").draw()?;
        chart.draw_series(LineSeries::new(history.iter().map(|(x,y)| (*x as u32,(*y).max(low))),color.stroke_width(2)))?;
    }
    root.present()?;
    Ok(())
});
xor_figure

Evolved XOR network
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Truth-table MSE (log)
 
 
Objective evaluations
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
0.001
 
 
 
0.01
 
 
 
0.1
 
 
 
1.0
 
 
 
 
0
 
 
 
100
 
 
 
200
 
 
 
300
 
 
 
400
 
 
 
500
 
 
 
600
 
 
 
700
 
 
 
800
 
 
 
900
 
 
 
1000
 
 
 
 
Differentiated XOR network
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Truth-table MSE (log)
 
 
Full-batch gradient steps
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
0.001
 
 
 
0.01
 
 
 
0.1
 
 
 
1.0
 
 
 
 
0
 
 
 
100
 
 
 
200
 
 
 
300
 
 
 
400
 
 
 
500
 
 
 
600
 
 
 
700
 
 
 
800
 
 
 
900
 
 
 
1000
 
 
 
1100
 
 
<polyline fill="none" opacity="1" stroke="#008C78" stroke-width="2" points="580,93 583,96 587,96 591,96 594,96 598,96 602,96 605,96 609,96 613,96 616,96 620,96 624,96 627,96 631,97 635,97 638,98 642,99 646,101 649,104 653,107 657,111 661,116 664,123 668,131 672,140 675,149 679,159 683,168 686,176 690,184 694,190 697,197 701,202 705,208 708,213 712,217 716,221 719,225 723,228 727,232 730,235 734,238 738,241 742,243 745,246 749,248 753,251 756,253 760,255 764,257 767,259 771,261 775,262 778,264 782,266 786,267 789,269 793,270 797,272 800,273 804,275 808,276 811,277 815,279 819,280 823,281 826,282 830,283 834,284 837,285 841,287 845,288 848,289 852,290 856,291 859,291 863,292 867,293 870,294 874,295 878,296 881,297 885,298 889,298 892,299 896,300 900,301 904,302 907,302 911,303 915,304 918,304 922,305 926,306 929,306 933,307 937,308 940,308 944,309 948,310 951,310 955,311 959,312 962,312 966,313 970,313 973,314 977,314 981,315 985,316 "/>

## 5. Recurrent, convolutional and attention building blocks

These are checked forward computations. Only the MLP has analytic backpropagation in this milestone. RNN/LSTM state is explicit and belongs to the caller, making episode reset visible. Convolution here follows deep-learning **cross-correlation** conventions (no kernel reversal), with one input/output channel, unit stride and no padding.

In [11]:
let recurrent = RnnCell::new(1,1,vec![1.0,0.5,0.0])?;
let first_hidden = recurrent.step(&[1.0],&recurrent.zero_state())?;
let second_hidden = recurrent.step(&[0.0],&first_hidden)?;
println!("RNN hidden states: {first_hidden:?} then {second_hidden:?}");
let lstm = LstmCell::new(1,1,vec![0.0;12])?;
let lstm_state = lstm.step(&[1.0],&lstm.zero_state())?;
println!("Zero-parameter LSTM: hidden={:?}, cell={:?}",lstm_state.hidden,lstm_state.cell);
let image = vec![vec![1.0,2.0,3.0],vec![4.0,5.0,6.0],vec![7.0,8.0,9.0]];
let kernel = vec![vec![1.0,2.0],vec![3.0,4.0]];
println!("Valid cross-correlation: {:?}",conv2d_valid(&image,&kernel,0.5)?);

RNN hidden states: [0.7615941559557649] then [0.3633994843890525]


Zero-parameter LSTM: hidden=[0.0], cell=[0.0]


Valid cross-correlation: [[37.5, 47.5], [67.5, 77.5]]


Scaled dot-product attention computes `softmax(QKᵀ/√d)V`. Zero queries/keys produce uniform attention. A causal mask restricts each row to its prefix. The small `TransformerBlock` composes one attention head, residuals, normalization and a feedforward sublayer. It is a teaching block, not a complete pretrained model, multi-head implementation, or training framework. Positional encodings are added by the caller.

In [12]:
let queries = vec![vec![0.0];3];
let keys = queries.clone();
let values = vec![vec![2.0],vec![4.0],vec![6.0]];
let full = scaled_dot_product_attention(&queries,&keys,&values,false)?;
let causal = scaled_dot_product_attention(&queries,&keys,&values,true)?;
println!("Full attention values: {:?}",full.values);
println!("Causal attention values: {:?}",causal.values);
println!("Causal weights: {:?}",causal.weights);
let positions = sinusoidal_positions(3,4)?;
println!("First two position vectors: {:?}",&positions[..2]);
let transformer = TransformerBlock::new(4,8,vec![0.0;TransformerBlock::parameter_count(4,8)?])?;
println!("Zero-sublayer residual/normalization check: {:?}",transformer.forward(&positions,true)?);

Full attention values: [[4.0], [4.0], [4.0]]


Causal attention values: [[2.0], [3.0], [4.0]]


Causal weights: [[1.0, 0.0, 0.0], [0.5, 0.5, 0.0], [0.3333333333333333, 0.3333333333333333, 0.3333333333333333]]


First two position vectors: [[0.0, 1.0, 0.0, 1.0], [0.8414709848078965, 0.5403023058681398, 0.009999833334166664, 0.9999500004166653]]


Zero-sublayer residual/normalization check: [[-0.9999949998375027, 0.9999949998375027, -0.9999949998375027, 0.9999949998375027], [0.6451985726342562, -0.1526721641928172, -1.557575310135613, 1.065048901694174], [0.8888858850631619, -1.329636412003593, -0.5996181453416872, 1.0403686722821184]]


## 6. Extend one boundary at a time

- Implement `Evaluator<YourInput>` to introduce a new problem. Return a scalar, objective vector or your own result record.
- Implement `Algorithm` to change search state and its update rule. Keep the complete-batch contract explicit.
- Reuse the standalone operators in `nevo::operators` and Pareto functions in `nevo::multiobjective`.
- Add a tensor/backend adapter when batching or larger networks justify it. Burn and Bevy remain planned integrations; these reference implementations establish the behavior to test against.

Suggested experiments: change GA mutation probability; compare comma replacement with an explicitly implemented plus variant; examine raw versus centered-rank OpenAI ES; rotate an anisotropic objective for CMA-ES; increase ZDT1's budget; replace XOR's hidden width while keeping its parameter-count calculation automatic. Change one setting at a time and report evaluation budgets and all seeds.

### References and limits

- [Official book outline and assignments](https://neuroevolutionbook.com/).
- [Hansen, CMA-ES tutorial](https://arxiv.org/abs/1604.00772), classic positive-weight full-covariance variant.
- [Salimans et al., Evolution Strategies](https://arxiv.org/abs/1703.03864), mirrored-gradient estimator; this project uses a documented local SGD variant.
- [Deb et al., NSGA-II](https://doi.org/10.1109/4235.996017), nondominated sorting and crowding; variation is specified separately here.
- [Zitzler, Deb and Thiele, test problems](https://doi.org/10.1162/106365600568202), ZDT1.
- [Vaswani et al., Attention Is All You Need](https://arxiv.org/abs/1706.03762), attention and positional encodings.

See [coverage](../coverage.md) for implemented versus forward-only components, and [validation](../VALIDATION.md) for actual checks. The public HTML textbook is scheduled to return on January 28, 2027; this companion does not claim exhaustive coverage of unavailable chapter text or research-result reproduction.